# Week 7 — Why Does the Detector Say "AI"? Explanations, Error Analysis, Ethics

**Goal:** the fine-tuned DistilRoBERTa detector (`08`, thresholds from `09`) gives one number per essay. A student accused of using AI deserves to know *which parts* of their essay triggered it, and we need to know whether those explanations can be trusted. This notebook:

1. Builds two explanation methods that score every **sentence**: **occlusion** and **integrated gradients**.
2. **Tests them** instead of trusting them:
   - a *completeness* check: integrated-gradients attributions must add up to the score;
   - a *deletion* test: do the sentences a method calls "most AI" really move the score most?
   - a *splice* test: insert real AI sentences into a human essay; do the highlights find them?
3. Looks at the **global** picture: which tokens push essays towards "AI" across hundreds of essays.
4. **Error analysis:** who gets wrongly flagged, and which AI essays slip through.
5. **Ethics and limitations:** everything learned in Weeks 1–7, in one place.

The sentence-level scores feed the demo's highlighting (Week 8).

In [1]:
import os
import sys
sys.path.append('..')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import inspect
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from collections import defaultdict
from scipy.stats import spearmanr

from src.data import load_splits
from src.detector import Detector
import src.explain as explain
from src.explain import sentence_spans, occlusion, integrated_gradients

transformers.logging.set_verbosity_error()
SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 110)
HUMAN, AI = '#2a78d6', '#eb6834'
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e6e3', 'grid.linewidth': 0.6,
                     'axes.edgecolor': '#9a9a94', 'axes.titleweight': 'bold'})

det = Detector()
THR = det.thresholds[0.01]
data = load_splits()
print(f"detector loaded; decision threshold (1% FPR on val humans, from 09): logit {THR:.2f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

detector loaded; decision threshold (1% FPR on val humans, from 09): logit 4.67


## Step 1 — Method A: sentence occlusion

The most direct question: *what happens to the score if this sentence weren't there?* For each sentence, delete it, re-score the essay, and record

$$\text{contribution}_i = \text{logit(full essay)} - \text{logit(essay without sentence } i)$$

Positive means the sentence pushed the essay towards "AI". It costs one extra forward pass per sentence (~20 per essay), which is cheap. Its weakness: it measures each sentence **alone**. If two sentences each carry the same AI signal, removing either one changes little ("redundancy hides importance").

The detector only reads the first 512 tokens, so sentences after that point can't influence it. They get no score (`NaN`) rather than a misleading zero. The code, from `src/explain.py`:

In [2]:
print(inspect.getsource(explain.sentence_spans))
print(inspect.getsource(explain.occlusion))

def sentence_spans(text):
    """Character spans (start, end) of each sentence. A sentence ends at . ! or ? followed by whitespace."""
    spans, start = [], 0
    for m in re.finditer(r"(?<=[.!?])\s+", text):
        spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    return [(s, e) for s, e in spans if text[s:e].strip()]

def occlusion(det, text):
    """Per sentence: logit(full essay) - logit(essay without that sentence)."""
    spans = sentence_spans(text)
    limit = visible_chars(det, text)
    variants, idx = [], []
    for i, (s, e) in enumerate(spans):
        if s >= limit:
            continue
        cut_end = spans[i + 1][0] if i + 1 < len(spans) else len(text)   # remove the sentence and the space after it
        variants.append(text[:s] + text[cut_end:])
        idx.append(i)
    scores = det.score([text] + variants)
    out = np.full(len(spans), np.nan)
    out[idx] = scores[0] - scores[1:]
   

## Step 2 — Method B: integrated gradients

A gradient says how the score would change if a token's embedding moved slightly, but only *at that point*, and for a saturated classifier (logit ≈ 10) the gradient near the input is almost flat. **Integrated gradients** (Sundararajan et al., 2017) instead walks along a straight line from a **baseline** (here: the same essay with every word replaced by `<pad>`) to the real essay, averages the gradients along the way, and multiplies by how far each token's embedding moved:

$$\text{IG}_j = (x_j - b_j) \cdot \frac{1}{m}\sum_{k=1}^{m} \nabla_{x_j} F\!\left(b + \tfrac{k - 0.5}{m}(x - b)\right)$$

summed over the 768 embedding dimensions to get one number per token, then summed over the tokens of each sentence.

Its key property is **completeness**: all attributions add up exactly to $F(x) - F(b)$, the score of the essay minus the score of the baseline. That gives a built-in check that the integral was approximated well enough.

Why not just show the transformer's **attention weights**? Attention says where the model *looked*, not what *changed its answer*. Attention weights don't add up to the prediction, and several studies show they can be changed without changing the output. Integrated gradients is an attribution of the output itself.

In [3]:
print(inspect.getsource(explain.integrated_gradients))

def integrated_gradients(det, text, steps=128, chunk=16):
    """Token attributions by integrated gradients, summed per sentence.

    Baseline: the same essay with every real token replaced by <pad> (start/end tokens kept).
    Path: straight line in embedding space from baseline to the essay, `steps` midpoint samples.
    Returns sentence spans, per-sentence attribution, per-token attribution, token offsets, and the
    completeness check: attributions should sum to logit(essay) - logit(baseline).
    """
    model, tok, dev = det.model, det.tok, det.device
    enc = tok(text, truncation=True, max_length=det.max_len, return_offsets_mapping=True, return_tensors="pt")
    ids = enc["input_ids"].to(dev)
    offsets = enc["offset_mapping"][0].tolist()
    emb = model.roberta.embeddings.word_embeddings
    special = (ids == tok.bos_token_id) | (ids == tok.eos_token_id)
    base_ids = torch.where(special, ids, torch.full_like(ids, tok.pad_token_id))
    mask = torch.ones_like(ids)
    for 

**Completeness check: how many steps does the integral need?** On three AI essays, the gap between the summed attributions and $F(x) - F(b)$, relative to $F(x) - F(b)$. If the steps are too coarse, the attributions are simply wrong, and a highlight built on them would be fiction.

In [4]:
conv_texts = data.query("split == 'test' and on_topic and label == 1").sample(3, random_state=SEED)['text'].tolist()
rows = []
for steps in [16, 32, 64, 128]:
    rel, t0 = [], time.time()
    for txt in conv_texts:
        *_, c = integrated_gradients(det, txt, steps=steps)
        rel.append(abs(c['gap']) / abs(c['logit_essay'] - c['logit_baseline']))
    rows.append({'steps': steps, 'mean relative completeness error %': np.mean(rel) * 100, 'seconds per essay': (time.time() - t0) / 3})
pd.DataFrame(rows).set_index('steps').round(2)

,mean relative completeness error %,seconds per essay
steps,,
16,68.31,0.33
32,20.87,0.37
64,3.16,0.70
128,0.06,1.38


With 16–32 steps the attributions can be off by tens of percent: the logit rises steeply somewhere along the path, and coarse steps miss it. **128 steps** brings the error to ~0.1% (the default in `src/explain.py`). We also checked that fp16 vs fp32 makes no difference here; the number of steps is what matters.

**A property of the baseline:** an all-`<pad>` essay already scores as strongly *human* (logit ≈ −9). So for a human essay, $F(x) - F(b) \approx 0$, and the attributions sum to roughly nothing. Integrated gradients here answers *"what pushes this essay towards AI, compared with an empty essay?"*. That's the right question for highlighting suspicious sentences, but it can't explain why an essay looks human.

## Step 3 — One essay, both methods

An on-topic AI essay from `test`, sentence by sentence. Bars show each method's score; the two most "AI" sentences per method are marked.

In [5]:
def show_sentences(text, k=2, width=110):
    spans, occ, full = occlusion(det, text)
    _, ig, tok_attr, offsets, comp = integrated_gradients(det, text)
    top_occ = set(np.argsort(-np.nan_to_num(occ, nan=-1e9))[:k]); top_ig = set(np.argsort(-np.nan_to_num(ig, nan=-1e9))[:k])
    print(f"essay logit {full:.2f} ({'FLAGGED as AI' if full >= THR else 'not flagged'})  |  IG completeness gap {comp['gap']:+.3f}")
    print(f"{'occl':>6} {'IG':>6}   sentence   (* = top-{k} by that method)")
    for i, (s, e) in enumerate(spans):
        mark = ('O' if i in top_occ else ' ') + ('I' if i in top_ig else ' ')
        o = '   n/a' if np.isnan(occ[i]) else f"{occ[i]:6.2f}"
        g = '   n/a' if np.isnan(ig[i]) else f"{ig[i]:6.2f}"
        print(f"{o} {g} {mark} {text[s:e][:width]}")
    valid = ~np.isnan(occ) & ~np.isnan(ig)
    print(f"\nrank agreement between the two methods (Spearman): {spearmanr(occ[valid], ig[valid]).statistic:.2f}")
    return tok_attr, offsets

example = data.query("split == 'test' and on_topic and label == 1").sample(1, random_state=7)['text'].iloc[0]
tok_attr, offsets = show_sentences(example)

essay logit 9.75 (FLAGGED as AI)  |  IG completeness gap +0.044
  occl     IG   sentence   (* = top-2 by that method)
  0.00   0.45 O  In recent years, there has been a growing movement to limit car usage.
  0.00   0.63    This movement is based on the belief that cars are harmful to the environment, unhealthy for individuals, and 
  0.00   0.63    There are many advantages to limiting car usage, and these advantages should be considered by all citizens.
  0.00   0.84    One of the main advantages of limiting car usage is that it is good for the environment.
  0.00   0.63    Cars produce a significant amount of pollution, which contributes to climate change and other environmental pr
  0.00   0.41    When people limit their car usage, they are helping to reduce pollution and protect the environment.
  0.00   0.81    Another advantage of limiting car usage is that it is good for personal health.
  0.00   0.88    Cars are a major source of sedentary behavior, which can lead to obesity, h

And the **tokens** inside the essay with the largest integrated-gradients attributions, towards AI and towards human:

In [6]:
toks = [example[s:e] for s, e in offsets]
order = np.argsort(tok_attr)
fmt = lambda idx: ', '.join(f"'{toks[i].strip()}' ({tok_attr[i]:+.2f})" for i in idx if toks[i].strip())
print("pushes towards AI:   ", fmt(order[::-1][:15]))
print("pushes towards human:", fmt(order[:10]))

pushes towards AI:    'conclusion' (+0.32), 'course' (+0.25), 'main' (+0.25), 'Finally' (+0.24), 'Overall' (+0.22), 'associated' (+0.22), 'concerted' (+0.21), 'instead' (+0.20), 'challenge' (+0.17), 'In' (+0.17), ',' (+0.16), ',' (+0.16), 'challenges' (+0.15), 'challenge' (+0.15), 'challenges' (+0.14)
pushes towards human: '.' (-0.01), '.' (-0.01), '.' (-0.00)


## Step 4 — Deletion test: are the explanations faithful?

An explanation is **faithful** if the parts it calls important really drive the prediction. The test: for flagged AI essays, delete the top-ranked sentences according to each method and re-score. A faithful method should make the logit fall **faster** than deleting the same number of random sentences.

**Why fractions, not 1–3 sentences:** the smoke test of this notebook showed that deleting 1–3 sentences barely moves the logit for *any* method (drops of 0.01–0.05). The detector's classification head ends in a `tanh`, so its logit has a ceiling near 9.8, and a confident AI essay sits right at it. Removing a few sentences from an essay that is AI-like throughout leaves it at the ceiling. So we delete **25%, 50% and 75%** of each essay's sentences, enough to bring it off the ceiling.

Notes for a fair reading:
- Only essays the detector reads completely (≤ 512 tokens), so every sentence has a score.
- Random deletion is averaged over 5 random draws per essay.

In [7]:
def remove_sentences(text, spans, drop):
    keep = [i for i in range(len(spans)) if i not in set(drop)]
    return ' '.join(text[spans[i][0]:spans[i][1]] for i in keep)

fully_read = lambda t: len(det.tok(t)['input_ids']) <= det.max_len
pool = data.query("split == 'test' and on_topic and label == 1")
pool = pool[pool['text'].map(fully_read)].sample(150, random_state=SEED)
pool = pool[det.score(pool['text']) >= THR]

t0 = time.time()
records = []
for txt in pool['text']:
    spans, occ, full = occlusion(det, txt)
    _, ig, *_ = integrated_gradients(det, txt)
    variants, labels = [], []
    for frac in [0.25, 0.5, 0.75]:
        k = max(1, int(round(frac * len(spans))))
        for name, ranking in [('occlusion', np.argsort(-occ)), ('integrated gradients', np.argsort(-ig))]:
            variants.append(remove_sentences(txt, spans, ranking[:k])); labels.append((name, frac))
        for r in range(5):
            variants.append(remove_sentences(txt, spans, rng.choice(len(spans), k, replace=False))); labels.append(('random', frac))
    sc = det.score(variants)
    for (name, frac), s in zip(labels, sc):
        records.append({'method': name, 'share of sentences deleted': frac, 'drop': full - s, 'still flagged': s >= THR})
deletion = (pd.DataFrame(records).groupby(['method', 'share of sentences deleted'])
            .agg(mean_logit_drop=('drop', 'mean'), still_flagged_pct=('still flagged', 'mean')))
deletion['still_flagged_pct'] *= 100
print(f"{len(pool)} flagged AI essays, {(time.time() - t0) / 60:.1f} min")
deletion.round(2).unstack('share of sentences deleted')

149 flagged AI essays, 3.4 min


mean_logit_drop             still_flagged_pct              
share of sentences deleted            0.25  0.50  0.75              0.25   0.50   0.75
method                                                                                
integrated gradients                  0.03  0.07  0.26             100.0  100.0  100.0
occlusion                             0.05  0.09  0.20             100.0  100.0  100.0
random                                0.02  0.04  0.10             100.0  100.0  100.0

## Step 5 — Splice test: do the highlights find real AI sentences?

A direct test of what the demo will show. Take an on-topic **human** essay, and insert **3 sentences from an AI essay on the same prompt** at random positions. Then:
- Is the spliced essay flagged at all? (Three AI sentences in an otherwise human essay is a light touch.)
- Does each method rank the 3 inserted sentences at the top? Measured as **precision@3**: the share of a method's top-3 sentences that are the inserted ones. Random guessing scores $3/n$ for an essay with $n$ sentences.

(This tests the *highlighting*, not a mixed-authorship detector: decision 7 keeps paragraph-level mixed-authorship detection out of scope.)

In [8]:
humans = data.query("split == 'test' and on_topic and label == 0")
humans = humans[humans['text'].map(lambda t: len(det.tok(t)['input_ids']) <= 380)]          # leave room for 3 more sentences
ais = data.query("split == 'test' and on_topic and label == 1")
ai_sents = defaultdict(list)
for _, r in ais.iterrows():
    for s, e in sentence_spans(r['text']):
        if len(r['text'][s:e].split()) >= 8:
            ai_sents[r['prompt_name']].append(r['text'][s:e])

t0 = time.time()
rows = []
for _, h in humans.sample(400, random_state=SEED).iterrows():
    if len(ai_sents[h['prompt_name']]) < 3:
        continue
    hs = [h['text'][s:e] for s, e in sentence_spans(h['text'])]
    ins = list(rng.choice(ai_sents[h['prompt_name']], 3, replace=False))
    positions = sorted(rng.choice(len(hs) + 3, 3, replace=False))
    merged, it_h, it_a = [], iter(hs), iter(ins)
    for p in range(len(hs) + 3):
        merged.append(next(it_a) if p in positions else next(it_h))
    text = ' '.join(merged)
    if len(det.tok(text)['input_ids']) > det.max_len:
        continue
    spans, occ, full = occlusion(det, text)
    _, ig, *_ = integrated_gradients(det, text)
    truth = set(positions)
    rows.append({'n_sentences': len(spans), 'logit': full, 'flagged': full >= THR,
                 'p@3 occlusion': len(set(np.argsort(-occ)[:3]) & truth) / 3,
                 'p@3 integrated gradients': len(set(np.argsort(-ig)[:3]) & truth) / 3,
                 'p@3 random': 3 / len(spans)})
    if len(rows) == 200:
        break
splice = pd.DataFrame(rows)
print(f"{len(splice)} spliced essays in {(time.time() - t0) / 60:.1f} min | flagged as AI: {splice['flagged'].mean() * 100:.1f}%")
for subset, d in [('all spliced essays', splice), ('only the ones flagged as AI', splice[splice['flagged']])]:
    print(f"\n{subset} (n={len(d)}):")
    print(d[['p@3 occlusion', 'p@3 integrated gradients', 'p@3 random']].mean().mul(100).round(1).to_string())

200 spliced essays in 3.7 min | flagged as AI: 3.0%

all spliced essays (n=200):
p@3 occlusion               64.3
p@3 integrated gradients    14.5
p@3 random                  22.8

only the ones flagged as AI (n=6):
p@3 occlusion               44.4
p@3 integrated gradients    33.3
p@3 random                  20.2


## Step 6 — Global view: which tokens push essays towards "AI"?

Integrated gradients on 200 AI and 200 human on-topic `test` essays, then for every token that occurs at least 30 times across them, its **mean attribution per occurrence**. That gives the detector's overall "vocabulary of suspicion", to compare with the strongest TF-IDF terms from `04` ("additionally", "in conclusion", "potential", "significant", …).

In [9]:
sample = pd.concat([data.query("split == 'test' and on_topic and label == 1").sample(200, random_state=SEED),
                    data.query("split == 'test' and on_topic and label == 0").sample(200, random_state=SEED)])
t0 = time.time()
tok_sum, tok_n = defaultdict(float), defaultdict(int)
for txt in sample['text']:
    _, _, attr, offs, _ = integrated_gradients(det, txt)
    for a, (s, e) in zip(attr, offs):
        w = txt[s:e].strip().lower()
        if w and any(ch.isalpha() for ch in w):
            tok_sum[w] += a; tok_n[w] += 1
glob = pd.DataFrame({'mean_attribution': {w: tok_sum[w] / tok_n[w] for w in tok_sum}, 'count': tok_n})
glob = glob[glob['count'] >= 30].sort_values('mean_attribution')
print(f"{len(sample)} essays in {(time.time() - t0) / 60:.1f} min; {len(glob)} tokens seen >= 30 times")
print("\nmost AI-pushing tokens:  ", ', '.join(glob.index[::-1][:25]))
print("\nmost human-pushing tokens:", ', '.join(glob.index[:25]))

400 essays in 8.3 min; 629 tokens seen >= 30 times

most AI-pushing tokens:   essay, conclusion, particularly, finally, additionally, furthermore, third, course, main, addition, overall, challenges, significant, including, argue, however, important, hand, plus, rather, argument, contribute, instead, impact, especially

most human-pushing tokens: very, due, nt, dont, ven, text, paragraph, l, isn, ent, his, back, ial, him, bog, bad, se, gives, tell, nasa, og, says, rac, go, surface


## Step 7 — Error analysis

Score every on-topic essay in `test`, `heldout_prompt` and `heldout_generator` at the 1% threshold, then describe the mistakes:
- **False positives** (human essays flagged as AI): how do they differ from other human essays in length, misspelling rate (share of words outside GloVe's vocabulary, as in `10`) and GPT-2 perplexity (`10`)?
- **False negatives** (AI essays not flagged): which sources?

In [10]:
import gensim.downloader as gensim_api
from gensim.utils import simple_preprocess
glove_vocab = set(gensim_api.load('glove-wiki-gigaword-100').key_to_index)
gpt2 = pd.read_csv('../results/10_gpt2_features.csv', index_col='row_id')

ev = data[data['split'].isin(['test', 'heldout_prompt', 'heldout_generator']) & data['on_topic']].join(gpt2[['log_ppl']]).copy()
ev['logit'] = det.score(ev['text'])
ev['flagged'] = ev['logit'] >= THR
ev['misspell_pct'] = [np.mean([w not in glove_vocab for w in simple_preprocess(t, min_len=1, max_len=30)] or [0]) * 100 for t in ev['text']]

h = ev[ev['label'] == 0]
fp = h[h['flagged']]
print(f"human essays: {len(h):,} | flagged (false positives): {len(fp)} ({len(fp) / len(h) * 100:.2f}%)")
print(pd.DataFrame({'false positives (median)': fp[['n_words', 'misspell_pct', 'log_ppl']].median(),
                    'all other humans (median)': h[~h['flagged']][['n_words', 'misspell_pct', 'log_ppl']].median()}).round(2))
print("\nfalse-positive rate by split and prompt (%):")
print(h.groupby(['split', 'prompt_name'])['flagged'].agg(['size', 'mean']).assign(mean=lambda d: (d['mean'] * 100).round(2))
      .rename(columns={'size': 'humans', 'mean': '% flagged'}).to_string())

human essays: 6,692 | flagged (false positives): 191 (2.85%)
              false positives (median)  all other humans (median)
n_words                         489.00                     441.00
misspell_pct                      0.00                       0.56
log_ppl                           2.92                       3.29

false-positive rate by split and prompt (%):
                                                      humans  % flagged
split          prompt_name                                             
heldout_prompt Does the electoral college work?         2674       1.31
               Summer projects                          1746       7.73
test           "A Cowboy Who Rode the Waves"             132       0.00
               Car-free cities                           246       1.22
               Cell phones at school                     163       0.61
               Community service                         140       1.43
               Distance learning                     

In [11]:
a = ev[ev['label'] == 1]
fn = a[~a['flagged']]
print(f"AI essays: {len(a):,} | missed (false negatives): {len(fn)} ({len(fn) / len(a) * 100:.2f}%)\n")
print(a.groupby(['split', 'source'])['flagged'].agg(['size', 'mean']).assign(missed=lambda d: ((1 - d['mean']) * 100).round(1))
      [['size', 'missed']].rename(columns={'size': 'AI essays', 'missed': '% missed'}).to_string())

AI essays: 4,502 | missed (false negatives): 7 (0.16%)

                                                      AI essays  % missed
split             source                                                 
heldout_generator darragh_claude_v6                         832       0.6
                  darragh_claude_v7                         791       0.0
                  falcon_180b_v1                            425       0.0
heldout_prompt    NousResearch/Llama-2-7b-chat-hf           200       0.0
                  chat_gpt_moth                              10       0.0
                  cohere-command                             50       0.0
                  darragh_claude_v6                         113       0.0
                  darragh_claude_v7                         134       0.0
                  falcon_180b_v1                             52       0.0
                  kingki19_palm                             672       0.0
                  llama2_chat                           

Two false positives and two false negatives, read with their sentence-level occlusion scores: what made the detector wrong?

In [12]:
def brief(text, n=6):
    spans, occ, full = occlusion(det, text)
    order = np.argsort(-np.nan_to_num(occ, nan=-1e9))
    print(f"   logit {full:.2f} | sentences most responsible for the 'AI' score:")
    for i in order[:3]:
        print(f"      {occ[i]:+.2f}  {text[spans[i][0]:spans[i][1]][:150]}")

for title, rows in [("FALSE POSITIVES (human essays flagged as AI)", fp.nlargest(2, 'logit')),
                    ("FALSE NEGATIVES (AI essays not flagged)", fn.nsmallest(2, 'logit'))]:
    print(f"===== {title} =====")
    for _, r in rows.iterrows():
        print(f"\n[{r['split']} | {r['source']} | {r['prompt_name']} | {r['n_words']} words | misspellings {r['misspell_pct']:.1f}%]")
        print("  ", r['text'][:300].replace('\n', ' '), '...')
        brief(r['text'])
    print()

===== FALSE POSITIVES (human essays flagged as AI) =====

[heldout_prompt | persuade_corpus | Summer projects | 334 words | misspellings 0.0%]
   In order to assure that students are learning over the summer, some schools require students to complete summer projects during their break. Some people believe that summer projects should be student-designed, while others believe that summer projects should be teacher-designed. All in all, summer p ...
   logit 9.68 | sentences most responsible for the 'AI' score:
      +0.02  In conclusion, summer projects should be student-designed.
      +0.02  Some people believe that summer projects should be student-designed, while others believe that summer projects should be teacher-designed.
      +0.02  Secondly, if summer projects are student-designed, then students won't be as stressed about the project as they would if it were the teacher's project

[test | persuade_corpus | Car-free cities | 175 words | misspellings 0.0%]
   In our modern age, 

   logit -7.91 | sentences most responsible for the 'AI' score:
      +0.17  However, the dangers of cell phone use outweigh any benefits.
      -0.25  Laws banning phones are necessary to achieve an optimal level of safety on public roads.
      -0.31  There is significant evidence that cell phone use severely impairs a driver's ability to pay full attention to the road.

[test | mistral7binstruct_v2 | Distance learning | 457 words | misspellings 0.0%]
   It is often said that you can not make a child, teenager, or adult do what they do not want to do. Imagine starting a new school year; students are ready to learn, to take classes they will really enjoy, and to see their friends. It comes time to pick up their schedule and they see that there is som ...
   logit -6.99 | sentences most responsible for the 'AI' score:
      +0.17  Having good grades is what makes your GPA great.
      +0.04  We all know that having good grades and keeping a good GPA is very important when it comes to c

## Step 8 — Save

In [13]:
deletion.round(3).to_csv('../results/14_deletion_test.csv')
splice.round(3).to_csv('../results/14_splice_test.csv', index=False)
glob.round(4).to_csv('../results/14_global_tokens.csv')
ev[['split', 'label', 'source', 'prompt_name', 'logit', 'flagged', 'n_words', 'misspell_pct', 'log_ppl']].to_csv('../results/14_errors.csv')
print("saved")

saved


## Key takeaways

- **The detector's verdict is spread across the whole essay, not concentrated in a few sentences.** For a confident AI essay, removing even **75% of its sentences** leaves it flagged in 100% of cases (149 essays); the logit drops by at most ~0.26 from its ceiling near 9.8. Its classification head ends in a `tanh`, so the logit saturates, and when the AI style runs through every sentence, no single sentence is decisive.
- **That makes occlusion blind on clear-cut AI essays.** Leave-one-sentence-out scores every sentence ≈ 0.00 on the worked example: removing any one sentence leaves the essay at the ceiling. Integrated gradients still separates sentences there (completeness error 0.06% at 128 steps).
- **Integrated gradients is blind on essays that look human.** The all-`<pad>` baseline already scores as human (logit ≈ −9), so for a human-looking essay the attributions sum to ~0 and are mostly noise. In the splice test it found the inserted AI sentences **less often than random** (precision@3 14.5% vs 22.8%).
- **Occlusion finds inserted AI sentences well when the rest of the essay is human:** precision@3 **64%** vs 23% for random. But only **3%** of essays with 3 spliced-in AI sentences were flagged at all, so the detector judges whole essays. A few AI sentences in a human essay aren't enough to trip it, which matches decision 7 (mixed authorship is out of scope).
- **Both methods are more faithful than random, but the effect is small in absolute terms.** Deleting the top 75% of sentences by integrated gradients drops the logit 0.26 vs 0.10 for random; occlusion 0.20. Occlusion is slightly ahead at 25% and 50%.
- **What the detector reacts to (integrated gradients over 400 essays):** AI side: "essay", "conclusion", "particularly", "finally", "additionally", "furthermore", "overall", "challenges", "significant", "however": discourse markers and formal vocabulary, the same as TF-IDF's top terms in `04`. Human side: "very", "dont", "isn", "nt" (contractions without apostrophes), references to the source text ("text", "paragraph", "nasa"), and **fragments of misspelled words** ("ven", "ent", "ial", "og").
- **Errors at the 1%-FPR threshold (on-topic test + held-out slices):** 191 of 6,692 human essays flagged (**2.85%**), up to **7.7%** on the unseen "Summer projects" prompt. 7 of 4,502 AI essays missed (0.16%). The false positives are longer (median 489 vs 441 words), have **no misspellings** (0.00% vs 0.56%) and lower GPT-2 perplexity (2.92 vs 3.29): **careful, fluent student writers**.
- **For the demo's highlighting:** use occlusion when it shows contrast between sentences. When the essay is flagged and occlusion is flat (saturated), switch to integrated gradients and say so ("AI-like style throughout; these sentences contribute most"). Never present sentence highlights as proof: the decision is about the whole essay.


## Ethics and limitations

Everything below comes from measurements in notebooks `03`–`14`.

**What the detector really learned**
- DAIGT-V2 pits **student essays** (US grades 6–12, Persuade corpus) against **2023-era LLM essays**. The detector separates *student writing* from *LLM writing*. It has never seen fluent adult human writing labelled "human", so it has no basis for judging it.
- Its strongest cues are **discourse markers** ("in conclusion", "additionally", "furthermore") and **formal vocabulary** on the AI side, and **misspellings, missing apostrophes and informal phrasing** on the human side (`14`, consistent with `04`, `05`, `10`). These are stylistic correlates, not proof of authorship.

**Who gets hurt by mistakes**
- **The false positives are the best student writers.** Flagged human essays are longer, have essentially no misspellings and read more fluently (`09`, `10`, `14`). A student taught to write formal, well-structured essays with transition words is exactly the profile the detector flags. Non-native writers who learned formal templates are plausibly affected the same way (not measured here; there is no such group label in the data).
- **The false-positive rate is not stable.** The threshold was set for 1% on familiar prompts, yet 7.7% of human essays on the unseen "Summer projects" prompt are flagged (`09`, `14`). No threshold fixed in advance guarantees the error rate on a new assignment.
- **Using a paraphrasing tool on honest work raises the flag rate** (0.7% → 2.7% after T5 paraphrasing, small sample; `13`).

**What it cannot do**
- It cannot say **which model** wrote an essay: attribution collapses to chance on a data source it hasn't seen (`11`).
- It cannot reliably flag an essay that is only **partly** AI-written: 3 AI sentences in a human essay were flagged 3% of the time (`14`).
- Its robustness was tested only against typos and light/medium paraphrasing (`13`). It was **not** tested against LLM "humaniser" rewriting or against models released after 2023.
- Perplexity-based detection (`10`) fails on stronger generators (Claude, GPT-4 read about as "surprising" as humans), which is a warning about how quickly detectors age.

**Explanations are not evidence**
- Sentence highlights are *approximate*: each method is blind in one regime (`14`), and both beat random by only a modest margin in the deletion test. They show what the model reacted to, not what the student did.

**Responsible use**
- This is a course project. It **must not** be used to accuse or grade real students.
- If a detector like this were ever used, a flag should start a conversation (drafts, revision history, an oral check), never decide an outcome, and the error rates above (especially for strong writers and new prompts) should be disclosed to anyone affected.
